In [ ]:
import csv

def calculate_average(values):
    """Return the average of a list of numeric values."""
    if not values:
        return None
    return sum(values) / len(values)

def load_numeric_values_from_csv(file_path):
    """Load all numeric values from a CSV file into a list."""
    values = []

    with open(file_path, mode='r', newline='') as file:
        reader = csv.reader(file)

        for row in reader:
            for item in row:
                try:
                    values.append(float(item))
                except ValueError:
                    pass

    return values

# Load the CSV file and calculate the average
numbers = load_numeric_values_from_csv('data.csv')
average = calculate_average(numbers)

print('Numeric values:', numbers)
print('Average:', average)

In [ ]:
import csv
from sklearn.tree import DecisionTreeClassifier

def to_number(value):
    """Convert a value to float when possible."""
    try:
        return float(value)
    except ValueError:
        return None

def load_csv_rows(file_path):
    """Load a CSV file as a list of dictionaries."""
    with open(file_path, mode='r', newline='', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        return list(reader), reader.fieldnames

def encode_features(rows, feature_columns):
    """Convert numeric and text feature values into numbers for sklearn."""
    category_maps = {column: {} for column in feature_columns}
    encoded_rows = []

    for row in rows:
        encoded_row = []

        for column in feature_columns:
            value = row[column].strip()
            number = to_number(value)

            if number is not None:
                encoded_row.append(number)
            else:
                if value not in category_maps[column]:
                    category_maps[column][value] = len(category_maps[column])
                encoded_row.append(category_maps[column][value])

        encoded_rows.append(encoded_row)

    return encoded_rows

def classify_csv_items(file_path, class_column=None):
    """Train a classifier from the CSV and predict the final class for each item."""
    rows, columns = load_csv_rows(file_path)

    if not rows:
        print('No rows found in the CSV file.')
        return []

    if class_column is None:
        class_column = columns[-1]

    feature_columns = [column for column in columns if column != class_column]
    labeled_rows = [row for row in rows if row[class_column].strip()]

    if not labeled_rows:
        print(f'No class labels found in column: {class_column}')
        return []

    X_train = encode_features(labeled_rows, feature_columns)
    y_train = [row[class_column].strip() for row in labeled_rows]
    X_all = encode_features(rows, feature_columns)

    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_all)

    for index, predicted_class in enumerate(predictions, start=1):
        print(f'Item {index}: {predicted_class}')

    return predictions

# Assumption: the last column in data.csv contains the class/label.
final_classes = classify_csv_items('data.csv')